In [17]:
import torch
import numpy as np

from src.nano_maker.skeleton import SkeletonModel
from src.nano_maker.container import configs

from src.nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

In [2]:
c = configs.skeleton_config
model = SkeletonModel(n_embd=c['n_embd'], n_head=c['n_head'], n_layers=c['n_layers'],
                      block_size=c['block_size'], map4_res=c['map4_res'], max_angstroms=c['max_angstroms'],dropout=c['dropout'])

In [3]:
from src.paths import *
sk_wt_path = PROJECT_ROOT / "src/nano_maker/container/skeleton_2.pt"

In [4]:
prototype_weights = torch.load(sk_wt_path, map_location="cpu")

In [5]:
print(type(prototype_weights))

<class 'dict'>


In [6]:
model.load_state_dict(prototype_weights["model_state_dict"])

<All keys matched successfully>

In [7]:
device = 'cpu'
block_size = c["block_size"]
radial_resolution = c["radial_resolution"]

In [8]:
sample_smiles = "CCNC(=O)[C@H]1CCCN1C(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](Cc1ccc(O)cc1)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)CCc1ccc(Cl)cc1 |wU:57.63,12.20,wD:45.57,31.44,23.28,5.4,63.67,(30.68,-5.02,;30.68,-6.54,;29.36,-7.34,;28.03,-6.58,;26.73,-7.34,;28,-5.06,;29.2,-4.18,;28.69,-2.75,;27.18,-2.75,;26.75,-4.2,;25.41,-4.94,;25.41,-6.48,;24.09,-4.18,;24.09,-2.64,;25.44,-1.88,;25.44,-.33,;26.79,.42,;26.79,1.95,;28.13,2.73,;25.47,2.74,;22.76,-4.94,;21.41,-4.16,;21.44,-2.62,;20.08,-4.9,;20.06,-6.45,;21.4,-7.24,;21.38,-8.77,;22.73,-6.47,;18.74,-4.13,;17.42,-4.89,;17.41,-6.42,;16.1,-4.12,;16.1,-2.59,;15.64,-1.13,;14.17,-.64,;14.17,.9,;15.65,1.37,;16.26,2.77,;17.81,2.93,;18.71,1.7,;18.07,.28,;16.54,.12,;14.77,-4.87,;13.43,-4.1,;13.43,-2.55,;12.08,-4.86,;12.08,-6.38,;13.4,-7.18,;13.4,-8.72,;14.72,-9.49,;16.07,-8.73,;17.39,-9.51,;16.07,-7.19,;14.75,-6.42,;10.76,-4.07,;9.44,-4.83,;9.41,-6.38,;8.11,-4.06,;8.12,-2.52,;9.44,-1.77,;6.77,-4.81,;5.44,-4.03,;5.45,-2.51,;4.1,-4.8,;4.09,-6.34,;5.42,-7.12,;6.83,-6.5,;7.85,-7.64,;7.06,-8.98,;7.53,-10.44,;6.5,-11.57,;5,-11.24,;4.52,-9.78,;5.57,-8.66,;2.78,-4.03,;1.43,-4.77,;1.45,-6.31,;.08,-4.03,;-1.37,-4.99,;-2.89,-4.2,;-2.98,-2.49,;-4.53,-1.72,;-5.95,-2.65,;-7.46,-1.8,;-5.86,-4.38,;-4.33,-5.15,)|"

sample_map4 = torch.tensor(smiles_fingerprint(sample_smiles), dtype=torch.float32).unsqueeze(0)
sample_skeleton = model.generate(sample_map4)
sample_skeleton

[tensor([[47.3015, 55.6182, 53.0316]], grad_fn=<MulBackward0>),
 tensor([[29.7842, 28.0326, 32.0145]], grad_fn=<MulBackward0>),
 tensor([[33.6264, 43.9008, 46.4491]], grad_fn=<MulBackward0>),
 tensor([[40.3502, 48.6927, 53.9637]], grad_fn=<MulBackward0>),
 tensor([[40.5613, 44.0106, 45.8146]], grad_fn=<MulBackward0>),
 tensor([[35.5539, 31.9292, 32.8807]], grad_fn=<MulBackward0>),
 tensor([[31.0567, 49.4841, 50.5141]], grad_fn=<MulBackward0>),
 tensor([[36.4560, 50.6485, 50.2153]], grad_fn=<MulBackward0>),
 tensor([[ 8.1244, 12.5294, 11.6641]], grad_fn=<MulBackward0>),
 tensor([[16.8310, 40.2215, 45.7947]], grad_fn=<MulBackward0>),
 tensor([[15.9587, 45.1391, 45.5320]], grad_fn=<MulBackward0>),
 tensor([[14.6341, 43.8844, 40.3389]], grad_fn=<MulBackward0>),
 tensor([[14.1272, 42.1473, 46.8195]], grad_fn=<MulBackward0>),
 tensor([[11.4479, 44.9960, 50.1043]], grad_fn=<MulBackward0>),
 tensor([[11.5714, 39.0957, 41.9478]], grad_fn=<MulBackward0>),
 tensor([[ 9.9337, 42.9947, 42.5252]], g

In [9]:
model.eval()
with torch.no_grad():
    ctx = torch.zeros(1, block_size, 3, device=device)

    smiles_list = [
        ("Aspirin", "CC(=O)Oc1ccccc1C(=O)O"),
        ("Caffeine", "Cn1cnc2c1c(=O)n(c(=O)n2C)C"),
        ("Ibuprofen", "CC(C)Cc1ccc(cc1)C(C)C(=O)O"),
    ]

    for name, smi in smiles_list:
        fp = torch.tensor(smiles_fingerprint(smi),
                         dtype=torch.float32).unsqueeze(0).to(device)
        out, _ = model(ctx, fp)
        print(f"{name}: {out.squeeze().tolist()}")

Aspirin: [26.5561466217041, 31.341243743896484, 33.572872161865234]
Caffeine: [24.657556533813477, 35.42039108276367, 38.2028694152832]
Ibuprofen: [27.879362106323242, 36.65312957763672, 38.04367446899414]


In [11]:
# Pocket Skeleton Generation algorithm prototyping

In [12]:
from src.nano_maker.modules.nAAno.radialseeker import RadialSeeker

In [13]:
r_mod = RadialSeeker(100, 33, False)

In [14]:
def process_skeleton(pocket_skeleton):
    output = []
    for vector in pocket_skeleton:
        output.append(vector.detach().flatten().tolist())
    return output

In [15]:
processed_sample = process_skeleton(sample_skeleton)
processed_sample

[[47.301536560058594, 55.61821365356445, 53.03155517578125],
 [29.78416633605957, 28.032590866088867, 32.014549255371094],
 [33.626434326171875, 43.9007682800293, 46.449134826660156],
 [40.3502311706543, 48.692684173583984, 53.963714599609375],
 [40.56132888793945, 44.010597229003906, 45.81461715698242],
 [35.55393981933594, 31.929208755493164, 32.88072204589844],
 [31.056669235229492, 49.48414993286133, 50.51414108276367],
 [36.45601272583008, 50.64847946166992, 50.21534729003906],
 [8.124425888061523, 12.5293607711792, 11.664058685302734],
 [16.83095359802246, 40.221458435058594, 45.79467010498047],
 [15.95867919921875, 45.13908386230469, 45.53203582763672],
 [14.634062767028809, 43.88443374633789, 40.33891677856445],
 [14.127242088317871, 42.14729309082031, 46.819461822509766],
 [11.447893142700195, 44.99599075317383, 50.10434341430664],
 [11.571434020996094, 39.09568786621094, 41.94776153564453],
 [9.933720588684082, 42.994720458984375, 42.52518844604492],
 [8.44343376159668, 42.00

In [16]:
# convert everything into raw angstrom coordinates
angstrom_output = []
for vector in processed_sample:
    angstrom_output.append(r_mod.radial_to_xyz(vector))
angstrom_output

[[-2.7809858703613273, 2.7080210113525425, 1.0008264160156273],
 [-14.342450218200682, -15.498490028381347, -12.870397491455076],
 [-11.806553344726563, -5.025492935180662, -3.343571014404297],
 [-7.368847427368163, -1.862828445434566, 1.6160516357421884],
 [-7.229522933959959, -4.953005828857421, -3.7623526763916004],
 [-10.53439971923828, -12.926722221374511, -12.29872344970703],
 [-13.502598304748535, -1.3404610443115246, -0.6606668853759743],
 [-9.939031600952148, -0.5720035552978473, -0.8578707885742176],
 [-28.637878913879394, -25.730621891021727, -26.301721267700195],
 [-22.891570625305175, -7.453837432861327, -3.775517730712888],
 [-23.467271728515627, -4.208204650878905, -3.9488563537597656],
 [-24.341518573760986, -5.03627372741699, -7.376314926147458],
 [-24.676020221710203, -6.182786560058592, -3.0991551971435527],
 [-26.44439052581787, -4.302646102905271, -0.9311333465576155],
 [-26.362853546142578, -8.196846008300781, -6.314477386474607],
 [-27.443744411468504, -5.6234844

In [22]:
true_sample = []
for vector in angstrom_output:
    appending = True
    if abs(vector[0]) < 0.33:
        break
    true_sample.append(vector)

true_sample

[[-2.7809858703613273, 2.7080210113525425, 1.0008264160156273],
 [-14.342450218200682, -15.498490028381347, -12.870397491455076],
 [-11.806553344726563, -5.025492935180662, -3.343571014404297],
 [-7.368847427368163, -1.862828445434566, 1.6160516357421884],
 [-7.229522933959959, -4.953005828857421, -3.7623526763916004],
 [-10.53439971923828, -12.926722221374511, -12.29872344970703],
 [-13.502598304748535, -1.3404610443115246, -0.6606668853759743],
 [-9.939031600952148, -0.5720035552978473, -0.8578707885742176],
 [-28.637878913879394, -25.730621891021727, -26.301721267700195],
 [-22.891570625305175, -7.453837432861327, -3.775517730712888],
 [-23.467271728515627, -4.208204650878905, -3.9488563537597656],
 [-24.341518573760986, -5.03627372741699, -7.376314926147458],
 [-24.676020221710203, -6.182786560058592, -3.0991551971435527],
 [-26.44439052581787, -4.302646102905271, -0.9311333465576155],
 [-26.362853546142578, -8.196846008300781, -6.314477386474607],
 [-27.443744411468504, -5.6234844

In [23]:
print(len(true_sample))

38


In [ ]:
# would be good to track n_trainable parameters